# NB24 --- What IS the Concentric System?

## Premise

The thesis claims the concentric geometry describes the reality *behind* what we
observe --- a spiritual physics whose flat limit gives the natural world. Observable
physics (standard QM, Cartesian coordinates, 3+1 parsing) is what you get when
curvature goes to zero and coupling switches off.

This notebook asks: **what mathematical structure IS this system?**

NB23 showed the geometry has genuine logical structure (forced ordering, curvature
requirement, exponential recurrence gradient). But NB23 left two questions open:

1. Is the ~21x recurrence factor a genuine signature or just pi/tolerance?
2. Does {2,3,5,7} produce different behavior than other frequency sets?

This notebook resolves both questions and identifies the system's mathematical class.

### Four Parts

1. **Mathematical identity** --- What class of dynamical system is this?
2. **Recurrence decomposition** --- Separate the tolerance artifact from the coupling signal
3. **Frequency comparison** --- Do the primes matter, or does any ascending set work?
4. **What the coupling predicts** --- Corrections beyond flat-space physics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from itertools import combinations
import seaborn as sns

sns.set_theme(style='whitegrid')
%matplotlib inline

# -- Generic concentric system accepting arbitrary frequencies --

class GenericConcentricSystem:
    """
    Iterated skew-product on T^n with arbitrary base frequencies
    and hierarchical (inner->outer) coupling.

    ODE:
      d(theta_0)/dt = omega_0
      d(theta_k)/dt = omega_k + alpha * sum_{j<k} (1/f_j) * sin(theta_j)

    where omega_p = 2*pi*sqrt(f_p) and f_j are the frequency labels.
    """

    def __init__(self, freqs, alpha=0.3):
        self.freqs = np.asarray(freqs, dtype=float)
        self.n = len(self.freqs)
        self.omega = 2 * np.pi * np.sqrt(self.freqs)
        self.alpha = alpha
        self.weights = (1.0 / self.freqs) / (1.0 / self.freqs).sum()

    def ode(self, t, theta):
        dtheta = self.omega.copy()
        if self.alpha > 0:
            for k in range(1, self.n):
                for j in range(k):
                    dtheta[k] += self.alpha * (1.0 / self.freqs[j]) * np.sin(theta[j])
        return dtheta

    def integrate(self, t_span, n_points=200000, theta0=None):
        if theta0 is None:
            theta0 = np.zeros(self.n)
        t_eval = np.linspace(t_span[0], t_span[1], n_points)
        sol = solve_ivp(self.ode, t_span, theta0, t_eval=t_eval,
                        method='RK45', rtol=1e-10, atol=1e-12)
        return {'t': sol.t, 'theta': sol.y, 'theta_mod': np.mod(sol.y, 2*np.pi)}

    def jacobian(self, theta):
        """Jacobian df_i/dtheta_j at a given state."""
        J = np.zeros((self.n, self.n))
        for k in range(1, self.n):
            for j in range(k):
                J[k, j] = self.alpha * (1.0 / self.freqs[j]) * np.cos(theta[j])
        return J

print("GenericConcentricSystem ready.")

## Part 1: Mathematical Identity

The system's ODE has a very specific structure. Let's name it precisely.

In [ ]:
sys_primes = GenericConcentricSystem([2, 3, 5, 7], alpha=0.3)

# -- Property 1: Jacobian structure --
print("PROPERTY 1: JACOBIAN STRUCTURE")
print("=" * 60)
theta_test = np.array([1.0, 2.0, 3.0, 4.0])
J = sys_primes.jacobian(theta_test)
print(f"\nJacobian at theta = {theta_test}:")
print(np.array2string(J, precision=4, suppress_small=True))
print(f"\nUpper triangle (should be all zeros):")
upper = np.triu(J, k=0)  # includes diagonal
print(np.array2string(upper, precision=4, suppress_small=True))
is_lower = np.allclose(upper, 0)
print(f"\nStrictly lower-triangular: {'YES' if is_lower else 'NO'}")
print(f"Eigenvalues: {np.linalg.eigvals(J)}")
print(f"All eigenvalues zero: {'YES' if np.allclose(np.linalg.eigvals(J), 0) else 'NO'}")

In [ ]:
# -- Property 2: Divergence-free (volume-preserving) --
print("PROPERTY 2: DIVERGENCE")
print("=" * 60)
print()
print("For the ODE d(theta_k)/dt = f_k(theta):")
print("  div(f) = sum_k  df_k/dtheta_k")
print()
print("  f_0(theta) = omega_0             -> df_0/dtheta_0 = 0")
print("  f_k(theta) = omega_k + alpha*sum_{j<k}(1/f_j)*sin(theta_j)")
print("  df_k/dtheta_k = 0  (coupling depends on theta_j with j < k, not theta_k)")
print()
print("  div(f) = 0 for ALL states theta.")
print("  The flow is VOLUME-PRESERVING on T^4.")
print()
print("Meaning: the system preserves Liouville measure.")
print("No states created or destroyed -- conservation law.")

In [ ]:
# -- Property 3: NOT Hamiltonian --
print("PROPERTY 3: SYMPLECTICITY TEST")
print("=" * 60)
print()
print("A Hamiltonian system on T^4 requires: J^T * Omega + Omega * J = 0")
print("for some symplectic matrix Omega.")
print()

pairings = [
    ([0,1,2,3], "canonical: (theta_2,theta_3) x (theta_5,theta_7)"),
    ([0,2,1,3], "cross: (theta_2,theta_5) x (theta_3,theta_7)"),
    ([0,3,1,2], "extreme: (theta_2,theta_7) x (theta_3,theta_5)"),
]

any_hamiltonian = False
for perm, label in pairings:
    Omega = np.zeros((4, 4))
    Omega[0, 1] = 1; Omega[1, 0] = -1
    Omega[2, 3] = 1; Omega[3, 2] = -1
    J_reord = J[np.ix_(perm, perm)]
    test = J_reord.T @ Omega + Omega @ J_reord
    is_ham = np.allclose(test, 0, atol=1e-10)
    if is_ham:
        any_hamiltonian = True
    print(f"  Pairing {label}:")
    print(f"    Hamiltonian: {'YES' if is_ham else 'NO'}  (residual norm: {np.linalg.norm(test):.6f})")

print()
if not any_hamiltonian:
    print("NOT Hamiltonian for ANY symplectic structure on T^4.")
    print()
    print("WHY: Hamiltonian systems have TWO-WAY coupling (action = reaction).")
    print("This system has ONE-WAY coupling: inner -> outer only.")
    print("The inner orbit modulates the outer, but NOT vice versa.")
    print()
    print("THESIS INTERPRETATION: This is influx in mathematics.")
    print("Higher degrees (inner orbits) flow into lower degrees (outer).")
    print("The influence does not flow back.")
else:
    print("System IS Hamiltonian for at least one pairing.")

In [ ]:
# -- Classification --
print("MATHEMATICAL CLASSIFICATION")
print("=" * 60)
print()
print("The system is:")
print()
print("  1. A FLOW on T^4 (4-torus)")
print("  2. VOLUME-PRESERVING (divergence-free)")
print("  3. NOT HAMILTONIAN (one-way coupling)")
print("  4. STRICTLY HIERARCHICAL (lower-triangular Jacobian)")
print("  5. All Lyapunov exponents = 0 (ordered, not chaotic)")
print()
print("Mathematical name: ITERATED SKEW-PRODUCT OF CIRCLE ROTATIONS")
print()
print("This is a well-studied class in ergodic theory:")
print("  Start with base rotation: theta_2 -> theta_2 + omega_2*t")
print("  Extend to theta_3 via skew-product: depends on theta_2 only")
print("  Extend to theta_5: depends on theta_2, theta_3 only")
print("  Extend to theta_7: depends on theta_2, theta_3, theta_5 only")
print()
print("KEY: The filtration T^1 < T^2 < T^3 < T^4 is preserved.")
print("Lower levels are dynamically autonomous from higher levels.")
print("= 'influx flows downward, never upward.'")
print()
print("alpha = 0: fully decoupled (flat, Cartesian, proprium)")
print("alpha > 0: hierarchically coupled (curved, concentric, center)")

## Part 2: The Recurrence Ratio --- Geometry vs. Coupling

NB23 found a ~21x decay per nesting level in the joint recurrence count.
But is this a property of the system, or an artifact of the tolerance parameter?

For **independent** (uncoupled) oscillators, the fraction of time all k coordinates
are within tolerance delta of the origin is approximately (delta/pi)^k. The ratio
at each level is pi/delta. At delta = 0.15: pi/0.15 = 20.9.

**This matches the ~21x exactly.** So NB23's finding might just be this geometric
factor, telling us nothing about the coupling.

To separate the signals, we compare the recurrence ratio of the **coupled** system
(alpha > 0) against the **uncoupled** system (alpha = 0) at the same tolerance.
Any difference IS the coupling correction.

In [ ]:
def measure_recurrence(system, t_span=(0, 500), n_points=200000, tolerance=0.15):
    """Measure joint recurrence counts at each nesting depth."""
    result = system.integrate(t_span, n_points)
    theta_mod = result['theta_mod']
    initial = theta_mod[:, 0]
    counts = []
    for level in range(1, system.n + 1):
        near_mask = np.ones(theta_mod.shape[1], dtype=bool)
        for k in range(level):
            diff = np.abs(theta_mod[k, :] - initial[k])
            diff = np.minimum(diff, 2 * np.pi - diff)
            near_mask &= (diff < tolerance)
        near_mask[0] = False  # exclude initial point
        counts.append(int(near_mask.sum()))
    return counts

# Run at multiple tolerances for both coupled and uncoupled
tolerances = [0.05, 0.10, 0.15, 0.20, 0.30]
freqs = [2, 3, 5, 7]

print("RECURRENCE RATIO: COUPLED vs UNCOUPLED")
print("=" * 80)

def ratio_str(counts):
    parts = []
    for i in range(len(counts) - 1):
        if counts[i+1] > 0:
            parts.append(f"{counts[i]/counts[i+1]:.1f}")
        else:
            parts.append("inf")
    return " / ".join(parts)

print(f"{'delta':>6} {'pi/d':>8} | {'Uncoupled ratios':^30} | {'Coupled ratios':^30}")
print("-" * 80)

for delta in tolerances:
    sys_u = GenericConcentricSystem(freqs, alpha=0.0)
    sys_c = GenericConcentricSystem(freqs, alpha=0.3)
    counts_u = measure_recurrence(sys_u, tolerance=delta)
    counts_c = measure_recurrence(sys_c, tolerance=delta)
    pred = np.pi / delta
    print(f"{delta:>6.2f} {pred:>8.1f} | {ratio_str(counts_u):^30} | {ratio_str(counts_c):^30}")

print()
print("If coupled ratios ~= uncoupled ratios ~= pi/delta: coupling doesn't change geometry.")
print("If coupled ratios DIFFER from pi/delta: coupling produces a measurable correction.")

In [ ]:
# Detailed comparison at delta=0.15
delta = 0.15
pred = np.pi / delta

sys_u = GenericConcentricSystem(freqs, alpha=0.0)
sys_c = GenericConcentricSystem(freqs, alpha=0.3)
counts_u = measure_recurrence(sys_u, tolerance=delta)
counts_c = measure_recurrence(sys_c, tolerance=delta)

print(f"DETAILED COMPARISON AT delta = {delta}")
print("=" * 70)
print(f"  pi/delta prediction: {pred:.2f}")
print()
print(f"{'Level':>6} {'Uncoupled':>12} {'Coupled':>12} {'U ratio':>10} {'C ratio':>10} {'Diff':>8}")
print("-" * 65)

for k in range(len(freqs)):
    u_ratio = counts_u[k-1]/counts_u[k] if k > 0 and counts_u[k] > 0 else 0
    c_ratio = counts_c[k-1]/counts_c[k] if k > 0 and counts_c[k] > 0 else 0
    diff = c_ratio - u_ratio if k > 0 else 0
    r_u = f"{u_ratio:.2f}" if k > 0 else "-"
    r_c = f"{c_ratio:.2f}" if k > 0 else "-"
    d_str = f"{diff:+.2f}" if k > 0 else "-"
    print(f"  {k+1:>4} {counts_u[k]:>12} {counts_c[k]:>12} {r_u:>10} {r_c:>10} {d_str:>8}")

print()
if len(counts_u) >= 4 and counts_u[-1] > 0 and counts_c[-1] > 0:
    total_u = counts_u[0] / max(counts_u[-1], 1)
    total_c = counts_c[0] / max(counts_c[-1], 1)
    correction = (total_c - total_u) / total_u * 100
    print(f"  Total ratio (depth 1 / depth 4):")
    print(f"    Uncoupled: {total_u:.0f}")
    print(f"    Coupled:   {total_c:.0f}")
    print(f"    Correction: {correction:+.1f}%")

## Part 3: Do the Primes Matter?

This is the decisive test. We run the SAME dynamical system --- same coupling
structure, same alpha, same integration time --- with different frequency sets.

If {2,3,5,7} produces the same behavior as {4,6,10,14}, the specific primes
are irrelevant. If they produce different behavior, we need to understand WHY.

| Label | Frequencies | Notes |
|-------|------------|-------|
| **Primes** | {2, 3, 5, 7} | The thesis's primes |
| **Scaled** | {4, 6, 10, 14} | 2x primes (same ratios) |
| **Other primes** | {11, 13, 17, 19} | Different primes, larger, closer spacing |
| **Squares** | {4, 9, 25, 49} | p^2 values |
| **Consecutive** | {2, 3, 4, 5} | Three primes + one composite |
| **Irrationals** | {e, pi, sqrt2, phi} | No number-theoretic structure |

In [ ]:
# Define frequency sets
freq_sets = {
    'Primes {2,3,5,7}':       [2, 3, 5, 7],
    'Scaled {4,6,10,14}':     [4, 6, 10, 14],
    'Other primes {11-19}':   [11, 13, 17, 19],
    'Squares {4,9,25,49}':    [4, 9, 25, 49],
    'Consecutive {2,3,4,5}':  [2, 3, 4, 5],
    'Irrationals':            [np.e, np.pi, np.sqrt(2), (1+np.sqrt(5))/2],
}

alpha = 0.3
delta = 0.15
pred = np.pi / delta

print("FREQUENCY COMPARISON: RECURRENCE COUNTS")
print("=" * 95)
print(f"{'Set':>25} {'D1':>7} {'D2':>7} {'D3':>7} {'D4':>7} "
      f"{'R12':>7} {'R23':>7} {'R34':>7} {'Mean R':>8}")
print("-" * 95)

all_results = {}
for name, fs in freq_sets.items():
    sys = GenericConcentricSystem(fs, alpha=alpha)
    counts = measure_recurrence(sys, tolerance=delta)
    ratios = []
    for i in range(len(counts)-1):
        if counts[i+1] > 0:
            ratios.append(counts[i] / counts[i+1])
        else:
            ratios.append(np.inf)
    mean_r = np.mean([r for r in ratios if np.isfinite(r)]) if ratios else 0
    all_results[name] = {'counts': counts, 'ratios': ratios, 'mean_ratio': mean_r}

    r_strs = [f"{r:.1f}" if np.isfinite(r) else "inf" for r in ratios]
    print(f"{name:>25} {counts[0]:>7} {counts[1]:>7} {counts[2]:>7} {counts[3]:>7} "
          f"{r_strs[0]:>7} {r_strs[1]:>7} {r_strs[2]:>7} {mean_r:>8.1f}")

print(f"\n  pi/delta prediction: {pred:.1f}")
print(f"  If all mean R ~= {pred:.1f}, the primes don't affect recurrence geometry.")

In [ ]:
# -- Equidistribution speed --
# How quickly does each system fill T^4 uniformly?
# Circular variance: CV=1 means uniform, CV=0 means localized.

print("EQUIDISTRIBUTION SPEED")
print("=" * 80)

window_sizes = [5000, 10000, 50000, 100000]
print(f"{'Set':>25}", end="")
for w in window_sizes:
    print(f" {'t~'+str(w//400):>10}", end="")
print()
print("-" * 80)

equidist_results = {}
for name, fs in freq_sets.items():
    sys = GenericConcentricSystem(fs, alpha=alpha)
    result = sys.integrate((0, 500), n_points=200000)
    theta_mod = result['theta_mod']

    cvs = []
    for w in window_sizes:
        segment = theta_mod[:, :w]
        cv_per_orbit = []
        for k in range(sys.n):
            cv = 1.0 - np.abs(np.mean(np.exp(1j * segment[k, :])))
            cv_per_orbit.append(cv)
        cvs.append(np.mean(cv_per_orbit))

    equidist_results[name] = cvs
    print(f"{name:>25}", end="")
    for cv in cvs:
        print(f" {cv:>10.4f}", end="")
    print()

print()
print("Higher CV at early windows = FASTER equidistribution.")

In [ ]:
# -- Diophantine properties of frequency ratios --
# omega_i/omega_j = sqrt(f_i/f_j)
# The quality of rational approximations determines equidistribution speed.
# "More irrational" = harder to approximate = faster filling of the torus.

print("DIOPHANTINE PROPERTIES OF FREQUENCY RATIOS")
print("=" * 80)

def continued_fraction(x, n_terms=12):
    """Compute continued fraction coefficients of x."""
    coeffs = []
    for _ in range(n_terms):
        a = int(np.floor(x))
        coeffs.append(a)
        frac = x - a
        if abs(frac) < 1e-12:
            break
        x = 1.0 / frac
    return coeffs

for name, fs in freq_sets.items():
    fs_arr = np.array(fs)
    print(f"\n{name}:")
    print(f"  {'Pair':>12} {'Ratio':>10} {'CF prefix':>35} {'MaxCF':>6}")
    print(f"  " + "-" * 68)

    gm_values = []
    for i, j in combinations(range(len(fs_arr)), 2):
        ratio = np.sqrt(fs_arr[j] / fs_arr[i])
        cf = continued_fraction(ratio)
        # Geometric mean of CF coefficients beyond a_0: proxy for approximability
        coeffs = [c for c in cf[1:] if c > 0]
        gm = np.exp(np.mean(np.log(np.array(coeffs, dtype=float)))) if coeffs else 1.0
        gm_values.append(gm)
        cf_str = str(cf[:8])
        fi = f"{fs_arr[i]:.3f}" if fs_arr[i] != int(fs_arr[i]) else str(int(fs_arr[i]))
        fj = f"{fs_arr[j]:.3f}" if fs_arr[j] != int(fs_arr[j]) else str(int(fs_arr[j]))
        max_cf = max(cf[1:]) if len(cf) > 1 else 0
        print(f"  {fi+'/'+fj:>12} {ratio:>10.6f} {cf_str:>35} {max_cf:>6}")

    print(f"  Mean GM(a_k): {np.mean(gm_values):.2f}  (LOWER = more irrational, faster equidist.)")

In [ ]:
# -- Connection curvature comparison --
# Measures how much inner angles twist per unit of outer angle change.
# This IS the coupling geometry made quantitative.

print("CONNECTION CURVATURE COMPARISON")
print("=" * 80)

def connection_curvature_generic(theta_mod, segment_length=200):
    """Curvature = std(d_inner/d_outer) per segment, averaged."""
    n_orbits = theta_mod.shape[0]
    n_points = theta_mod.shape[1]
    n_segments = n_points // segment_length
    results = {}
    for k in range(1, n_orbits):
        for j in range(k):
            curvatures = []
            for seg in range(n_segments):
                s = seg * segment_length
                e = s + segment_length
                dj = np.diff(theta_mod[j, s:e])
                dk = np.diff(theta_mod[k, s:e])
                mask = np.abs(dk) > 1e-8
                if mask.sum() > 10:
                    connection = dj[mask] / dk[mask]
                    curvatures.append(np.std(connection))
            results[(j, k)] = np.mean(curvatures) if curvatures else 0.0
    return results

pair_labels = ['0>1', '0>2', '0>3', '1>2', '1>3', '2>3']
print(f"{'Set':>25}", end="")
for pl in pair_labels:
    print(f" {pl:>8}", end="")
print(f" {'Total':>8}")
print("-" * 90)

curvature_totals = {}
for name, fs in freq_sets.items():
    sys = GenericConcentricSystem(fs, alpha=alpha)
    result = sys.integrate((0, 500), n_points=200000)
    curv = connection_curvature_generic(result['theta_mod'])
    total = sum(curv.values())
    curvature_totals[name] = total

    print(f"{name:>25}", end="")
    for k in range(1, 4):
        for j in range(k):
            print(f" {curv.get((j,k), 0):>8.4f}", end="")
    print(f" {total:>8.4f}")

print()
print("Higher total curvature = MORE structured (non-flat) dynamics.")

In [ ]:
# -- Inner state variance at angular returns --
# When orbit k returns to its starting angle, how much has
# inner content changed? Higher = more coupling effect.

print("INNER STATE VARIANCE AT RETURNS (COUPLING SIGNATURE)")
print("=" * 80)

def inner_variance_at_returns(theta_mod, tolerance=0.1):
    """For each orbit k, measure variance of inner orbits when k returns."""
    n_orbits = theta_mod.shape[0]
    variances = []
    for idx in range(n_orbits):
        theta_target = theta_mod[idx, 0]
        inner_states = []
        for i in range(1, theta_mod.shape[1]):
            diff = abs(theta_mod[idx, i] - theta_target)
            diff = min(diff, 2 * np.pi - diff)
            if diff < tolerance and idx > 0:
                inner_states.append(theta_mod[:idx, i].copy())
        if inner_states and idx > 0:
            inner_matrix = np.array(inner_states)
            variances.append(np.var(inner_matrix, axis=0).sum())
        else:
            variances.append(0.0)
    return variances

print(f"{'Set':>25} {'V(orb1)':>10} {'V(orb2)':>10} {'V(orb3)':>10} {'Monotone':>10}")
print("-" * 75)

for name, fs in freq_sets.items():
    sys = GenericConcentricSystem(fs, alpha=alpha)
    result = sys.integrate((0, 500), n_points=200000)
    vs = inner_variance_at_returns(result['theta_mod'])
    # Check monotonicity of variance in orbits 1,2,3
    mono = all(vs[i] <= vs[i+1] for i in range(1, len(vs)-1))
    tag = "YES" if mono else "NO"
    print(f"{name:>25} {vs[1]:>10.3f} {vs[2]:>10.3f} {vs[3]:>10.3f} {tag:>10}")

print()
print("If all sets show same profile: primes don't matter here.")
print("If primes differ: they produce a specific coupling geometry.")

## Part 4: What the Coupling Predicts Beyond Flat Space

The coupling parameter alpha controls departure from Cartesian (flat, independent)
physics. At alpha = 0, the system has independent orbits (standard QM). At alpha > 0,
it predicts **correlations between coordinates that should be independent**.

The strength and pattern of these correlations = the coupling correction.
If the correction has a specific signature for {2,3,5,7}, the primes predict
observable physics beyond the flat limit.

In [ ]:
# -- Cross-correlation of angular velocities --
# Uncoupled: orbits independent -> zero correlation
# Coupled: inner modulates outer -> nonzero correlation pattern

print("CROSS-CORRELATION MATRIX: COUPLED SYSTEMS")
print("=" * 80)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, (name, fs) in enumerate(freq_sets.items()):
    ax = axes.flat[idx]
    sys = GenericConcentricSystem(fs, alpha=alpha)
    result = sys.integrate((0, 500), n_points=200000)

    velocities = np.diff(result['theta_mod'], axis=1)
    # Handle wraparound
    velocities = np.where(velocities > np.pi, velocities - 2*np.pi, velocities)
    velocities = np.where(velocities < -np.pi, velocities + 2*np.pi, velocities)

    corr = np.corrcoef(velocities)
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-0.3, vmax=0.3)
    ax.set_title(name[:25], fontsize=9)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))

    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{corr[i,j]:.3f}", ha='center', va='center', fontsize=7)

plt.suptitle("Angular Velocity Cross-Correlations (alpha=0.3)", fontsize=13)
plt.tight_layout()
plt.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label='Correlation')
plt.show()

# Summary table
print(f"\n{'Set':>25} {'Total |corr|':>14} {'Max |corr|':>14}")
print("-" * 60)
for name, fs in freq_sets.items():
    sys = GenericConcentricSystem(fs, alpha=alpha)
    result = sys.integrate((0, 500), n_points=200000)
    velocities = np.diff(result['theta_mod'], axis=1)
    velocities = np.where(velocities > np.pi, velocities - 2*np.pi, velocities)
    velocities = np.where(velocities < -np.pi, velocities + 2*np.pi, velocities)
    corr = np.corrcoef(velocities)
    off_diag = corr[np.triu_indices(4, k=1)]
    print(f"{name:>25} {np.sum(np.abs(off_diag)):>14.4f} {np.max(np.abs(off_diag)):>14.4f}")

In [ ]:
# -- Coupling correction as function of alpha --
# Sweep alpha from 0 (Cartesian) to 0.5 (strong coupling)

alphas = np.linspace(0, 0.5, 15)

fig, ax = plt.subplots(figsize=(12, 7))

for name, fs in freq_sets.items():
    total_corrs = []
    for a in alphas:
        sys = GenericConcentricSystem(fs, alpha=a)
        result = sys.integrate((0, 300), n_points=100000)
        velocities = np.diff(result['theta_mod'], axis=1)
        velocities = np.where(velocities > np.pi, velocities - 2*np.pi, velocities)
        velocities = np.where(velocities < -np.pi, velocities + 2*np.pi, velocities)
        corr = np.corrcoef(velocities)
        off_diag = corr[np.triu_indices(4, k=1)]
        total_corrs.append(np.sum(np.abs(off_diag)))
    ax.plot(alphas, total_corrs, 'o-', label=name[:20], linewidth=2, markersize=4)

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='alpha=0 (Cartesian)')
ax.set_xlabel('Coupling strength alpha', fontsize=12)
ax.set_ylabel('Total |cross-correlation|', fontsize=12)
ax.set_title('Departure from Cartesian Independence', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("Slope at small alpha = 'coupling susceptibility' of each frequency set.")
print("Different slopes -> different frequency sets respond differently to coupling.")

## Summary

In [ ]:
from IPython.display import Markdown

lines = []
lines.append("## Results: Mathematical Structure of the Concentric System\n")

lines.append("### Part 1: Mathematical Identity\n")
lines.append("The system is an **iterated skew-product of circle rotations** on T^4:\n")
lines.append("| Property | Value | Meaning |")
lines.append("|----------|-------|---------|")
lines.append("| Divergence | 0 | Volume-preserving (Liouville measure conserved) |")
lines.append("| Hamiltonian? | NO | One-way coupling breaks symplecticity |")
lines.append("| Jacobian | Strictly lower-triangular | Hierarchical: inner autonomous from outer |")
lines.append("| Lyapunov exponents | All zero | Ordered, not chaotic |")
lines.append("| Coupling direction | Inner -> Outer only | = influx flows downward |")
lines.append("| Parameter alpha | 0 = Cartesian flat; >0 = concentric curved | Controls departure from observable physics |")
lines.append("")

lines.append("### Part 2: Recurrence Ratio Correction\n")
primes_mean = all_results[list(all_results.keys())[0]]['mean_ratio']
lines.append(f"The NB23 finding of ~21x per level at delta=0.15 matches pi/delta = {np.pi/0.15:.1f}.")
lines.append("The comparison between coupled and uncoupled reveals whether coupling")
lines.append("produces a quantitative correction to this geometric baseline.\n")

lines.append("### Part 3: Frequency Comparison\n")
lines.append("| Set | Mean Recurrence Ratio | Total Curvature | Equidist. Speed |")
lines.append("|-----|----------------------|-----------------|-----------------|")
for name in freq_sets:
    mr = all_results[name]['mean_ratio']
    ct = curvature_totals.get(name, 0)
    eq = equidist_results.get(name, [0,0,0,0])
    lines.append(f"| {name} | {mr:.1f} | {ct:.4f} | {eq[-1]:.4f} |")
lines.append("")

lines.append("### What This Establishes\n")
lines.append("1. **The system has a precise mathematical identity**: iterated skew-product. ")
lines.append("This is a known class in ergodic theory with specific properties.\n")
lines.append("2. **The coupling alpha is the key physical parameter**: at alpha=0, standard ")
lines.append("(Cartesian) physics. At alpha>0, hierarchical correlations emerge.\n")
lines.append("3. **The question 'do the primes matter?' has a testable answer**: the frequency ")
lines.append("comparison reveals whether {2,3,5,7} produces different curvature, correlations, ")
lines.append("and equidistribution than alternatives.\n")

lines.append("### What This Points Toward\n")
lines.append("If this is the spiritual physics behind what we see, then:\n")
lines.append("- **alpha** parametrizes depth of reality. alpha=0 is the natural plane ")
lines.append("(flat, Cartesian, independent). alpha>0 is the spiritual plane ")
lines.append("(curved, concentric, coupled).\n")
lines.append("- **The cross-correlations** at alpha>0 are invisible connections between ")
lines.append("coordinates that standard physics treats as independent. They are predictions ")
lines.append("of the theory.\n")
lines.append("- **The filtration** (inner autonomous from outer) is the mathematical form ")
lines.append("of influx: higher causes flow into lower effects, not vice versa.\n")
lines.append("- **The volume-preserving property** is conservation: nothing created or ")
lines.append("destroyed; total state space conserved under evolution.")

display(Markdown("\n".join(lines)))